In [3]:
import re
import polars as pl
import yaml

PARQUET_FILE = "spy_10k_2015_present.parquet"
LEXICON_FILE = "AI_DEI_Lexicon.yml"

START_YEAR = 2015
END_YEAR = 2024
TOP_K = 5
CHUNK_SIZE = 30

with open(LEXICON_FILE, "r", encoding="utf-8") as f:
    lex = yaml.safe_load(f)

AI_PATS = list(lex.get("ai") or [])
DEI_PATS = list(lex.get("dei") or [])

# What this does: firm-year docs across ALL sectors (entire report)
df = pl.read_parquet(PARQUET_FILE, columns=["cik", "gics_sector", "filing_period", "text"])

docs = (
    df.with_columns(
        pl.col("cik").cast(pl.Utf8),
        pl.col("gics_sector").cast(pl.Utf8),
        pl.col("filing_period").cast(pl.Date),
        pl.col("filing_period").dt.year().alias("year"),
        pl.col("text").cast(pl.Utf8).fill_null(""),
    )
    .filter(
        (pl.col("year") >= START_YEAR)
        & (pl.col("year") <= END_YEAR)
        & pl.col("cik").is_not_null()
        & pl.col("gics_sector").is_not_null()
    )
    .group_by(["cik", "year"])
    .agg(pl.col("text").str.concat("\n").alias("text"))
)

# What this does: script-level label cleaner so regex never leaks into reporting
def regex_to_label(pat: str) -> str:
    s = pat
    s = re.sub(r"^\(\?i\)", "", s)
    s = s.replace(r"\b", "")
    s = s.replace(r"[-\s]+", " ")
    s = s.replace(r"[-\s]", " ")
    s = s.replace("?:", "")
    s = re.sub(r"[()]", "", s)
    s = s.replace("s?", "s")
    s = re.sub(r"\{\d+,?\d*\}", "", s)
    s = re.sub(r"\s+", " ", s).strip()

    lower = s.lower()
    overrides = {
        "ai": "AI",
        "a.i.": "AI",
        "artificial intelligence": "artificial intelligence",
        "machine learning": "machine learning",
        "deep learning": "deep learning",
        "natural language processing": "natural language processing",
        "nlp": "NLP",
        "llms": "LLM",
        "llm": "LLM",
        "generative ai": "generative ai",
        "dei": "DEI",
        "diversity equity inclusion": "diversity and inclusion",
        "diversity equity and inclusion": "diversity and inclusion",
        "equal employment opportunity": "equal employment opportunity",
        "non discrimination": "non-discrimination",
    }
    return overrides.get(lower, lower)

# What this does: count total occurrences per regex pattern across ALL firm-year docs
def total_occurrences(patterns: list[str], bucket: str) -> pl.DataFrame:
    base = docs.select(["text"])
    rows = []

    for start in range(0, len(patterns), CHUNK_SIZE):
        chunk = patterns[start:start + CHUNK_SIZE]
        exprs = [pl.col("text").str.count_matches(p).sum().alias(f"t{i}") for i, p in enumerate(chunk)]
        out = base.select(exprs)

        for i, pat in enumerate(chunk):
            rows.append(
                {
                    "bucket": bucket,
                    "regex": pat,
                    "label": regex_to_label(pat),
                    "total_occurrences": int(out.item(0, i)),
                }
            )

    return (
        pl.DataFrame(rows)
          .filter(pl.col("total_occurrences") > 0)
          .sort("total_occurrences", descending=True)
    )

ai_freq = total_occurrences(AI_PATS, "AI")
dei_freq = total_occurrences(DEI_PATS, "DEI")

# What this does: pick top K by frequency, dedup labels
def top_k_labels(freq_df: pl.DataFrame, k: int) -> list[str]:
    seen = set()
    out = []
    for r in freq_df.iter_rows(named=True):
        lab = r["label"]
        if lab in seen:
            continue
        seen.add(lab)
        out.append(lab)
        if len(out) >= k:
            break
    return out

AI_TOP5 = top_k_labels(ai_freq, TOP_K)
DEI_TOP5 = top_k_labels(dei_freq, TOP_K)

print(f"Top {TOP_K} AI terms (all firm-years, {START_YEAR}-{END_YEAR}):")
for i, t in enumerate(AI_TOP5, 1):
    print(f"  {i}. {t}")

print(f"\nTop {TOP_K} DEI terms (all firm-years, {START_YEAR}-{END_YEAR}):")
for i, t in enumerate(DEI_TOP5, 1):
    print(f"  {i}. {t}")


C:\Users\ddddd\AppData\Local\Temp\ipykernel_15724\330652204.py:37: DeprecationWarning: `str.concat` is deprecated; use `str.join` instead. Note also that the default `delimiter` for `str.join` is an empty string, not a hyphen.
  .agg(pl.col("text").str.concat("\n").alias("text"))


Top 5 AI terms (all firm-years, 2015-2024):
  1. agents
  2. AI
  3. automation
  4. transparency
  5. artificial intelligence

Top 5 DEI terms (all firm-years, 2015-2024):
  1. equity
  2. diversity
  3. inclusion
  4. inclusive|inclusivity
  5. disability


We decide to force bigrams to prevent terms with too many possible search vectors

To reduce ambiguity in high-frequency single-word terms 
(e.g., ‘equity’, ‘agents’), we query Google Trends using anchored bigrams
(e.g., ‘AI agents’, ‘pay equity’), which preserves the underlying concept while increasing search intent specificity

AI Google Trends terms:

AI

artificial intelligence

software agents

process automation

model transparency

DEI Google Trends terms:

pay equity

workplace diversity

workplace inclusion

inclusive workplace

workplace accessibility

The selected Google Trends terms reveal markedly different attention dynamics for AI and DEI. AI-related searches evolve gradually, with early public interest concentrated in adjacent operational concepts such as process automation and software agents, before consolidating around explicit conceptual terminology (artificial intelligence) after 2021. This pattern indicates an evolutionary diffusion, in which AI emerges through practice before being named as a distinct technological category. By contrast, DEI-related searches display a far more discrete regime shift, with pay equity, workplace diversity, and workplace inclusion rising sharply around 2020–2021 and stabilising thereafter. Unlike AI, DEI attention is immediately framed in institutional and workplace terms, leaving little ambiguity in interpretation. Together, these dynamics suggest that AI discourse matures incrementally, while DEI discourse is catalysed by a clearer structural breakpoint.